# 05 – Model Comparison & SHAP Explainability Analysis
**Owner:** Member 5 (Evaluation and Explainability Lead)

1. Compares all models fairly using `src/evaluate.compare_models()` (selecting the best model by **Validation RMSE** and reporting unbiased **Test** metrics).
2. Applies **SHAP** (SHapley Additive exPlanations) to the best-performing model.
3. Generates and saves SHAP summary, bar, and dependence plots plus mean `|SHAP|` importance rankings.

*Scientific Caution:* SHAP explains how the trained model uses features to form predictions; it does not independently prove physical causation.

In [1]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import FIGURES_DIR, MODELS_DIR, PREDICTIONS_DIR, RANDOM_SEED, RESULTS_DIR
from src.evaluate import compare_models
from src.features import SELECTED_FEATURES, get_xy

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

C:\Users\nourh\.gemini\antigravity\scratch\mof-co2-uptake-prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_order = [
    "baseline_dummy",
    "baseline_linear",
    "svr",
    "random_forest",
    "xgboost",
    "ann",
    "ga_ann",
]
comp_df = compare_models(model_order=model_order)
print("=== Model Comparison (Sorted by Validation RMSE) ===")
print(comp_df.to_string(index=False))

best_key = comp_df.iloc[0]["model_key"]
best_name = comp_df.iloc[0]["Model"]
print(f"\nBest model selected by Validation RMSE: {best_name} ({best_key})")

=== Model Comparison (Sorted by Validation RMSE) ===
                Model       model_key  Validation R2  Validation RMSE  Validation MAE  Test R2  Test RMSE  Test MAE  Training Time (s)
              XGBoost         xgboost         0.7027           0.9152          0.6130   0.6510     0.8643    0.5964               4.88
        Random Forest   random_forest         0.6269           1.0252          0.6990   0.5835     0.9442    0.6583               6.09
        ANN (PyTorch)             ann         0.6098           1.0486          0.6975   0.5343     0.9985    0.6918              16.82
        GA-ANN Hybrid          ga_ann         0.5766           1.0922          0.7545   0.5531     0.9781    0.6879              12.13
                  SVR             svr         0.5541           1.1209          0.7573   0.5011     1.0335    0.7100               1.13
Baseline (Dummy Mean)  baseline_dummy        -0.0089           1.6860          1.2883  -0.0007     1.4636    1.1681               0.13
  

In [3]:
X_train_df, y_train, _ = get_xy("train")
X_test_df, y_test, _ = get_xy("test")

if best_key in ("xgboost", "random_forest"):
    model_file = MODELS_DIR / ("xgboost_best.joblib" if best_key == "xgboost" else "random_forest_best.joblib")
    pipe = joblib.load(model_file)
    pre = pipe.named_steps["preprocessor"]
    est = pipe.named_steps["model"]
    X_test_arr = pre.transform(X_test_df)
    X_test_proc = pd.DataFrame(X_test_arr, columns=SELECTED_FEATURES)
    explainer = shap.TreeExplainer(est)
    shap_values = explainer.shap_values(X_test_arr)
elif best_key == "svr":
    pipe = joblib.load(MODELS_DIR / "svr_best.joblib")
    pre = pipe.named_steps["preprocessor"]
    est = pipe.named_steps["model"]
    X_train_proc = pd.DataFrame(pre.transform(X_train_df), columns=SELECTED_FEATURES)
    X_test_proc = pd.DataFrame(pre.transform(X_test_df), columns=SELECTED_FEATURES)
    bg = shap.sample(X_train_proc, 100, random_state=RANDOM_SEED)
    explainer = shap.KernelExplainer(est.predict, bg)
    X_test_proc = X_test_proc.iloc[:200].copy()
    shap_values = explainer.shap_values(X_test_proc)
else:
    # PyTorch ANN or GA-ANN fallback via KernelExplainer
    import torch.nn as nn
    pre = joblib.load(MODELS_DIR / "ann_preprocessor.joblib")
    ckpt_path = MODELS_DIR / ("ga_ann_best.pt" if best_key == "ga_ann" else "ann_best.pt")
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    hidden_dims = ckpt.get("hidden_dims", ckpt["config"].get("hidden_dims"))
    dropout = ckpt["config"]["dropout"]
    layers = []
    prev = len(SELECTED_FEATURES)
    for h in hidden_dims:
        layers.extend([nn.Linear(prev, h), nn.ReLU()])
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        prev = h
    layers.append(nn.Linear(prev, 1))
    net = nn.Sequential(*layers)
    state = {k.replace("net.", ""): v for k, v in ckpt["state_dict"].items()}
    net.load_state_dict(state)
    net.eval()
    def predict_fn(arr):
        with torch.no_grad():
            return net(torch.tensor(arr, dtype=torch.float32)).squeeze(-1).numpy()
    X_train_proc = pd.DataFrame(pre.transform(X_train_df), columns=SELECTED_FEATURES)
    X_test_proc = pd.DataFrame(pre.transform(X_test_df), columns=SELECTED_FEATURES).iloc[:200].copy()
    bg = shap.sample(X_train_proc, 100, random_state=RANDOM_SEED)
    explainer = shap.KernelExplainer(predict_fn, bg)
    shap_values = explainer.shap_values(X_test_proc)

# Save mean |SHAP| ranking
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({"feature": SELECTED_FEATURES, "mean_abs_shap": mean_abs_shap})
shap_df = shap_df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
shap_df.to_csv(RESULTS_DIR / "shap_feature_importance.csv", index=False)
print("Top 15 Features by Mean |SHAP|:")
print(shap_df.head(15).to_string(index=False))

Top 15 Features by Mean |SHAP|:
         feature  mean_abs_shap
mc_CRY-chi-0-all       0.282312
mc_CRY-chi-2-all       0.208610
   POAV [cm^3/g]       0.178601
              Di       0.166887
              Df       0.156086
  mc_CRY-S-1-all       0.117240
  PONAV [cm^3/g]       0.106155
  mc_CRY-S-0-all       0.097535
       f-lig-Z-1       0.093097
density [g/cm^3]       0.085019
  mc_CRY-Z-0-all       0.077512
  mc_CRY-I-1-all       0.062428
  mc_CRY-Z-1-all       0.058746
mc_CRY-chi-1-all       0.051843
D_mc_CRY-Z-1-all       0.045923


In [4]:
# 1. SHAP Summary Beeswarm Plot (Top 20)
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test_proc, max_display=20, show=False)
plt.title(f"SHAP Summary Plot – {best_name} (Top 20 Features)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_summary_beeswarm.png", dpi=200, bbox_inches="tight")
plt.close()

# 2. SHAP Bar Plot (Top 20)
plt.figure(figsize=(9, 6.5))
shap.summary_plot(shap_values, X_test_proc, plot_type="bar", max_display=20, show=False)
plt.title(f"SHAP Mean |SHAP| Feature Importance – {best_name}")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_bar_importance.png", dpi=200, bbox_inches="tight")
plt.close()

# 3. SHAP Dependence Plots for Top 3 Features (using raw unscaled test values on x-axis for interpretability)
top3_features = shap_df["feature"].head(3).tolist()
X_test_raw_sub = X_test_df.iloc[:len(X_test_proc)].reset_index(drop=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, feat in zip(axes, top3_features):
    f_idx = SELECTED_FEATURES.index(feat)
    sc = ax.scatter(
        X_test_raw_sub[feat],
        shap_values[:, f_idx],
        c=X_test_raw_sub[top3_features[1 if feat == top3_features[0] else 0]],
        cmap="coolwarm",
        alpha=0.55,
        s=18,
    )
    ax.axhline(0, color="black", linestyle="--", linewidth=1.0)
    ax.set_title(f"SHAP Dependence: {feat}")
    ax.set_xlabel(f"{feat} (original unit)")
    ax.set_ylabel("SHAP value (impact on predicted uptake)")
    fig.colorbar(sc, ax=ax, label=top3_features[1 if feat == top3_features[0] else 0])

fig.tight_layout()
fig.savefig(FIGURES_DIR / "shap_dependence_top3.png", dpi=200)
plt.close(fig)
print("Saved SHAP summary, bar, and top-3 dependence plots to figures/")

Saved SHAP summary, bar, and top-3 dependence plots to figures/
